# Fine-tune YOLO on Tooth Detection (PyTorch Lightning + W&B)

This notebook fine-tunes a pretrained YOLOv5 model on the tooth detection dataset using:
- Existing project data-preparation modules (`detection_pipeline`)
- A custom PyTorch Lightning training loop
- Weights & Biases logging for loss and mAP metrics

In [1]:
# If needed, uncomment to install dependencies in the active kernel.
# %pip install -q lightning wandb torchmetrics ultralytics

In [2]:
from __future__ import annotations

import os
import sys
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger

from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Ensure project imports work from notebook location.
PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config
from detection_pipeline import (
    DetectionDownloadConfig,
    ToothDetectionDataset,
    AugmentedToothDetectionDataset,
    build_detection_records,
    build_detection_train_pipeline,
    load_or_download_detection_dataset,
    split_grouped_records
)

In [ ]:
@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 32
    # Use single-process loading by default to avoid /dev/shm bus errors in containers.
    num_workers: int = 0
    max_epochs: int = 20
    lr: float = 0.00045636389024364654
    weight_decay: float = 0.000005312109908408371
    adamw_beta1: float = 0.9
    adamw_beta2: float = 0.999
    conf_threshold: float = 0.001
    iou_threshold: float = 0.6
    pretrained_weights: str = 'yolov5s.pt'
    wandb_project: str = 'tooth-detection-yolo-lightning'
    wandb_run_name: str = 'yolov5s-finetune'
    force_download: bool = False
    # Set to None to use the full dataset.
    train_subset_size: int | None = None
    val_subset_size: int | None = None
    test_subset_size: int | None = None

cfg = TrainConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda
Smoke test train/val sizes: 256 64


In [4]:
# Clone YOLOv5 if missing and put it first in import order for yolo-specific utils.
YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)

if not YOLOV5_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)
    ], check=True)

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

from models.yolo import Model
from utils.loss import ComputeLoss
from utils.general import intersect_dicts, non_max_suppression

print('YOLOv5 code loaded from:', YOLOV5_DIR)

YOLOv5 code loaded from: /work/external/yolov5


In [ ]:
class YoloTargetAdapterDataset(Dataset):
    """Wraps ToothDetectionDataset to emit labels compatible with YOLOv5 training loss."""

    def __init__(self, base_dataset: Dataset):
        self.base_dataset = base_dataset

    def __len__(self) -> int:
        return len(self.base_dataset)

    def __getitem__(self, index: int):
        image, target = self.base_dataset[index]
        return image, target


def yolo_collate_fn(batch: list[tuple[torch.Tensor, dict[str, Any]]]):
    images = []
    yolo_targets = []
    metric_targets = []

    for i, (img, tgt) in enumerate(batch):
        images.append(img.float())

        boxes_xyxy = tgt['boxes'].float()
        labels_one_based = tgt['labels'].long()

        # TorchMetrics expects class IDs starting at 0.
        metric_targets.append({
            'boxes': boxes_xyxy,
            'labels': (labels_one_based - 1).clamp_min(0),
        })

        if boxes_xyxy.numel() == 0:
            continue

        _, h, w = img.shape
        x1, y1, x2, y2 = boxes_xyxy[:, 0], boxes_xyxy[:, 1], boxes_xyxy[:, 2], boxes_xyxy[:, 3]

        cx = ((x1 + x2) * 0.5) / w
        cy = ((y1 + y2) * 0.5) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        cls = (labels_one_based - 1).float().clamp_min(0)
        batch_idx = torch.full((boxes_xyxy.shape[0],), float(i), dtype=torch.float32)

        packed = torch.stack([batch_idx, cls, cx, cy, bw, bh], dim=1)
        yolo_targets.append(packed)

    images = torch.stack(images, dim=0)
    if yolo_targets:
        yolo_targets = torch.cat(yolo_targets, dim=0)
    else:
        yolo_targets = torch.zeros((0, 6), dtype=torch.float32)

    return images, yolo_targets, metric_targets


class ToothDetectionDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_detection_dataset(
            DetectionDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download,
        )
        records, _ = build_detection_records(coco_data, image_dirs)

        train_rec, val_rec, test_rec = split_grouped_records(
            records,
            train_size=config.TRAIN_RATIO,
            val_size=config.VAL_RATIO,
            test_size=config.TEST_RATIO,
            random_state=42
        )

        if self.cfg.train_subset_size is not None:
            train_rec = train_rec[: self.cfg.train_subset_size]
        if self.cfg.val_subset_size is not None:
            val_rec = val_rec[: self.cfg.val_subset_size]
        if self.cfg.test_subset_size is not None:
            test_rec = test_rec[: self.cfg.test_subset_size]

        base_train = ToothDetectionDataset(
            train_rec, image_size=self.cfg.image_size, output_channels=3
        )
        aug_train = AugmentedToothDetectionDataset(base_train, build_detection_train_pipeline())
        
        base_val = ToothDetectionDataset(
            val_rec, image_size=self.cfg.image_size, output_channels=3
        )
        base_test = ToothDetectionDataset(
            test_rec, image_size=self.cfg.image_size, output_channels=3
        )

        self.train_ds = YoloTargetAdapterDataset(aug_train)
        self.val_ds = YoloTargetAdapterDataset(base_val)
        self.test_ds = YoloTargetAdapterDataset(base_test)

        print(f'Train/Val/Test sizes: {len(self.train_ds)}, {len(self.val_ds)}, {len(self.test_ds)}')

    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
            'collate_fn': yolo_collate_fn,
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            drop_last=True,
            **self._loader_kwargs(),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

In [ ]:
class LitYOLOv5(L.LightningModule):
    def __init__(self, cfg: TrainConfig, num_classes: int = 32):
        super().__init__()
        self.save_hyperparameters(ignore=["cfg"])

        self.cfg = cfg
        self.num_classes = num_classes
        self.compute_loss = None

        ckpt_path = Path(cfg.pretrained_weights)
        ckpt = torch.load(ckpt_path, map_location="cpu")

        yolo_cfg = ckpt["model"].yaml
        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()

        pretrained_state = ckpt["model"].float().state_dict()
        compatible_state = intersect_dicts(
            pretrained_state,
            self.model.state_dict(),
            exclude=["anchor"],
        )
        self.model.load_state_dict(compatible_state, strict=False)

        self.model.hyp = {
            "box": 0.05,
            "cls": 0.3,
            "obj": 0.7,
            "cls_pw": 1.0,
            "obj_pw": 1.0,
            "fl_gamma": 0.0,
            "label_smoothing": 0.0,
            "anchor_t": 4.0,
        }

        self.map_metric = MeanAveragePrecision(
            box_format="xyxy",
            class_metrics=False,
        )

    def on_train_start(self):
        self.model.to(self.device)
        self.compute_loss = ComputeLoss(self.model)
        self._fix_compute_loss_device()

    def _fix_compute_loss_device(self):
        if self.compute_loss is None:
            return

        self.compute_loss.device = self.device

        for name, value in vars(self.compute_loss).items():
            if torch.is_tensor(value):
                setattr(self.compute_loss, name, value.to(self.device))

            elif isinstance(value, list):
                setattr(
                    self.compute_loss,
                    name,
                    [
                        item.to(self.device) if torch.is_tensor(item) else item
                        for item in value
                    ],
                )

            elif isinstance(value, tuple):
                setattr(
                    self.compute_loss,
                    name,
                    tuple(
                        item.to(self.device) if torch.is_tensor(item) else item
                        for item in value
                    ),
                )

        if hasattr(self.compute_loss, "anchors"):
            self.compute_loss.anchors = self.compute_loss.anchors.to(self.device)

    def forward(self, x: torch.Tensor):
        return self.model(x)

    def training_step(self, batch, batch_idx: int):
        images, yolo_targets, _ = batch

        yolo_targets = yolo_targets.to(images.device, non_blocking=True)

        if self.compute_loss is None:
            self.compute_loss = ComputeLoss(self.model)

        self._fix_compute_loss_device()

        preds = self.model(images)
        loss, loss_items = self.compute_loss(preds, yolo_targets)

        self.log(
            "train/loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_box",
            loss_items[0],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_obj",
            loss_items[1],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_cls",
            loss_items[2],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )

        return loss

    def validation_step(self, batch, batch_idx: int):
        images, _, metric_targets = batch

        raw_output = self.model(images)

        if isinstance(raw_output, (tuple, list)):
            preds = raw_output[0]
        else:
            preds = raw_output

        nms_preds = non_max_suppression(
            preds,
            conf_thres=self.cfg.conf_threshold,
            iou_thres=self.cfg.iou_threshold,
            multi_label=False,
            max_det=300,
        )

        metric_preds = []

        for det in nms_preds:
            if det is None or len(det) == 0:
                metric_preds.append(
                    {
                        "boxes": torch.zeros((0, 4), device=self.device),
                        "scores": torch.zeros((0,), device=self.device),
                        "labels": torch.zeros(
                            (0,),
                            dtype=torch.long,
                            device=self.device,
                        ),
                    }
                )
                continue

            metric_preds.append(
                {
                    "boxes": det[:, :4],
                    "scores": det[:, 4],
                    "labels": det[:, 5].long(),
                }
            )

        metric_targets = [
            {
                "boxes": target["boxes"].to(self.device),
                "labels": target["labels"].to(self.device),
            }
            for target in metric_targets
        ]

        self.map_metric.update(metric_preds, metric_targets)

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()

        self.log("val/map", metrics["map"], prog_bar=True)
        self.log("val/map_50", metrics["map_50"], prog_bar=True)
        self.log("val/map_75", metrics["map_75"])

        self.map_metric.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay,
            betas=(self.cfg.adamw_beta1, self.cfg.adamw_beta2),
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.cfg.max_epochs,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

In [7]:
# Download pretrained YOLOv5 weights if missing.
weights_path = Path(cfg.pretrained_weights)
if not weights_path.exists():
    import urllib.request
    url = 'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt'
    urllib.request.urlretrieve(url, str(weights_path))

print('Using weights:', weights_path.resolve())

Using weights: /work/scripts/detection_pipeline/training/yolov5s.pt


In [ ]:
from lightning.pytorch.loggers import WandbLogger

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_path = PROJECT_ROOT / '.env'
if load_dotenv is not None and env_path.exists():
    load_dotenv(env_path, override=True)

wandb_key = (os.getenv('WANDB_API_KEY') or '').strip().strip('"').strip("'")
use_wandb = bool(wandb_key)
wandb_logger = False

if use_wandb:
    import wandb

    os.environ['WANDB_API_KEY'] = wandb_key
    wandb.login(key=wandb_key, relogin=True)
    wandb_logger = WandbLogger(
        project=cfg.wandb_project,
        name=cfg.wandb_run_name,
        log_model=True,
    )
    print('W&B enabled:', cfg.wandb_project, cfg.wandb_run_name)
else:
    print('WANDB_API_KEY not found. Running without W&B logging.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eznagyonkellettmz (eznagyonkellettmz-budapesti-m-szaki-s-gazdas-gtudom-nyi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful using WANDB_API_KEY from .env


In [ ]:
# Create data module and model
datamodule = ToothDetectionDataModule(cfg)
model = LitYOLOv5(cfg=cfg, num_classes=32)

print(f'Model created with image_size={cfg.image_size}, batch_size={cfg.batch_size}')
print(f'Max epochs: {cfg.max_epochs}, LR: {cfg.lr:.2e}, Weight decay: {cfg.weight_decay:.2e}')

In [ ]:
# Data sanity check
datamodule.setup('fit')

train_size = len(datamodule.train_ds)
val_size = len(datamodule.val_ds)
test_size = len(datamodule.test_ds)

print(f'\nDataset sizes:')
print(f'  Train: {train_size} samples')
print(f'  Val:   {val_size} samples')
print(f'  Test:  {test_size} samples')
print(f'  Total: {train_size + val_size + test_size} samples')

# Sample a few batches to check data loading
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()

print(f'\nDataloader batch sizes:')
print(f'  Train batch size: {cfg.batch_size}')
print(f'  Val batch size: {cfg.batch_size}')

# Load one batch to verify structure
images, yolo_targets, metric_targets = next(iter(train_loader))
print(f'\nBatch structure (from training dataloader):')
print(f'  Images shape: {images.shape}')
print(f'  YOLO targets shape: {yolo_targets.shape}')
print(f'  Metric targets count: {len(metric_targets)}')

if yolo_targets.numel() > 0:
    print(f'\nTarget format (batch_idx, class, cx, cy, w, h):')
    print(f'  First few targets:\n{yolo_targets[:5]}')
    classes_in_batch = yolo_targets[:, 1].unique().long()
    print(f'  Classes in batch: {sorted(classes_in_batch.tolist())}')

# Annotation statistics
total_annotations = 0
for i in range(min(len(datamodule.train_ds), 100)):
    _, target = datamodule.train_ds[i]
    total_annotations += len(target['labels'])

avg_annotations = total_annotations / min(len(datamodule.train_ds), 100)
print(f'\nAnnotation statistics (first 100 training samples):')
print(f'  Total annotations: {total_annotations}')
print(f'  Avg annotations per image: {avg_annotations:.2f}')

In [ ]:
# Callbacks
checkpoint_cb = ModelCheckpoint(
    dirpath=str(PROJECT_ROOT / 'output' / 'checkpoints'),
    filename='yolov5-finetune-{epoch:02d}-{val_map:.4f}',
    monitor='val/map',
    mode='max',
    save_top_k=3,
    save_last=True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

# Setup trainer
trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='auto',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    logger=wandb_logger if use_wandb else None,
    callbacks=[checkpoint_cb, lr_monitor],
    log_every_n_steps=200,
)

In [ ]:
# Train
print('Starting training...')
trainer.fit(model, datamodule=datamodule)

In [ ]:
# Run final validation
print('Running validation on full validation set...')
val_results = trainer.validate(model, datamodule=datamodule)
print(f'Validation results: {val_results}')

## Notes

- The dataset labels are converted from 1..32 to 0..31 for YOLO class indexing.
- Training and validation mAP are logged to W&B through Lightning's logger.
- You can adjust `TrainConfig` to tune the learning rate, batch size, and epochs.